In [1]:
# ============================================
# 0. 載入套件與環境設定
# ============================================
# 用途：將目前的資料夾路徑加入 Python 模組搜尋路徑，
#       以便 import 同一目錄下的自訂模組（train_save）。

import os
import sys

# 取得目前工作目錄（notebook 所在位置）
current_dir = os.getcwd()
# 若該路徑尚未存在於 sys.path，則插入到最前面，確保可正確載入自訂模組
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# 從 train_save.py 匯入「訓練並儲存模型」的函式
from train_save import train_and_save_model

# 輸出確認訊息，表示環境已準備完成
print("環境準備完畢，已成功載入 train_and_save_model 模組。")

環境準備完畢，已成功載入 train_and_save_model 模組。


Part 1：定義請求與回應的 Pydantic 模型

In [2]:
# ============================================
# 定義 Pydantic 訓練相關模型（與 app.py 相同）
# ============================================
# Pydantic 可用於定義資料結構並自動驗證欄位，
# 此處的模型與 FastAPI 的 app.py 保持一致，供 API 請求與回應使用。

from pydantic import BaseModel,Field
from pprint import pprint

# TrainConfig：訓練請求的輸入參數模型
class TrainConfig(BaseModel):
    test_size: float = Field(0.2, description="測試集分割比例", ge=0.1 , le=0.5) # 測試資料比例，範圍限制在 0.1 ~ 0.5
    random_state: int = Field(76, description="隨機種子", ge=0)                    # 隨機種子，固定後可重現相同結果
    model_type: str = Field("LinearRegression", description="模型演算法類型 (LinearRegression, Lasso, Ridge)") # 可選擇的迴歸演算法
    alpha: float = Field(1.0, description="正則化強度 alpha (適用於 Lasso 與 Ridge)", ge= 0.001, le=100.0)     # Lasso / Ridge 的正則化強度，範圍 0.001 ~ 100

# TrainResult：訓練完成後的回應資料模型
class TrainResult(BaseModel):
    status: str = Field(..., description="執行結果狀態")                # 執行結果（success / error）
    r2: float = Field(..., description="測試集 R-squared 決定係數")     # 決定係數，評估模型解釋力
    coef: list[float] = Field(..., description="特徵權重係數列表")      # 各特徵的迴歸係數（依順序排列）
    intercept: float = Field(..., description="截距")                   # 迴歸模型的截距
    feature_coefs: dict[str, float] = Field(..., description="特徵及其權重映射") # 特徵名稱對應係數的字典
    model_type: str = Field(..., description="模型演算法類型")          # 本次使用的演算法
    alpha: float = Field(..., description="正則化強度 alpha")           # 實際使用的正則化強度
    train_time: float = Field(..., description="訓練耗時 (秒)")         # 訓練所花費的時間
    message:str = Field(..., description="提示訊息")                    # 給使用者的提示訊息

# 印出模型的 JSON Schema，檢查欄位定義是否符合預期
print("TranConfig(BaseModel)")
pprint(TrainConfig.model_json_schema())
print("==============================")
print("TrainResult(BaseModel)")
pprint(TrainResult.model_json_schema())

TranConfig(BaseModel)
{'properties': {'alpha': {'default': 1.0,
                          'description': '正則化強度 alpha (適用於 Lasso 與 Ridge)',
                          'maximum': 100.0,
                          'minimum': 0.001,
                          'title': 'Alpha',
                          'type': 'number'},
                'model_type': {'default': 'LinearRegression',
                               'description': '模型演算法類型 (LinearRegression, '
                                              'Lasso, Ridge)',
                               'title': 'Model Type',
                               'type': 'string'},
                'random_state': {'default': 76,
                                 'description': '隨機種子',
                                 'minimum': 0,
                                 'title': 'Random State',
                                 'type': 'integer'},
                'test_size': {'default': 0.2,
                              'description': '測試集分割比例',
              

拆解 train_and_save_model() 底層訓練與序列化

In [3]:
# 拆解 train_and_save_model() 底層訓練與序列化
# 直接呼叫底層函式，指定以 Ridge 嶺迴歸進行訓練，
# 並傳入測試集比例、隨機種子與正則化強度 alpha。
from train_save import train_and_save_model
res_ridge:dict = train_and_save_model(
    test_size=0.2,         # 測試集佔 20%
    random_state=76,       # 隨機種子固定為 76，確保結果可重現
    model_type="Ridge",    # 使用 Ridge 嶺迴歸
    alpha=10.0             # 正則化強度為 10.0
)
# 以較易閱讀的格式印出訓練結果
pprint(res_ridge)

開始訓練 Ridge 嶺迴歸(α=10.0) (測試集比例:0.2, 隨機種子:76)....
正在將模型、預處理器與元數據序列化並儲存至 c:\Users\User\Documents\GitHub\2027-07-03clone\backend\2026_08_07\salary_model.joblib...
模型儲存成功！
{'alpha': 10.0,
 'coef': [3.915606705818322,
          10.029103401270465,
          -1.4644383465780102,
          -1.182860911975303,
          2.1482340576072554],
 'feature_coefs': {'City_城市A': -1.4644383465780102,
                   'City_城市B': -1.182860911975303,
                   'City_城市C': 2.1482340576072554,
                   'EducationLevel': 10.029103401270465,
                   'YearsExperience': 3.915606705818322},
 'intercept': 51.228571428571435,
 'message': 'Ridge 嶺迴歸(α=10.0) 模型訓練完成並儲存成功！',
 'model_type': 'Ridge',
 'r2': 0.8253872705107945,
 'status': 'success',
 'train_time': 0.09456443786621094}


理解 load_model_state() 全域動態更新機制

In [4]:
# 理解 load_model_state() 全域動態更新機制
# 目的：把已儲存的模型檔案重新載入到全域變數 MODEL_STATE，
#       讓服務在重新訓練後能即時使用最新的模型。
import joblib

# 組出模型檔案的完整路徑
current_dir = os.getcwd()
model_path = os.path.join(current_dir, "salary_model.joblib")

# 全域變數：存放目前服務使用的模型與相關元件
MODEL_STATE = {}

def load_model_state():
    global MODEL_STATE          # 宣告本函式要修改全域變數
    # 若模型檔案不存在，則先重新訓練並儲存一份
    if not os.path.exists(model_path):
        train_and_save_model()

    # 讀取模型檔案中序列化儲存的資料
    model_data = joblib.load(model_path)
    # 先清空舊的狀態，再更新為最新的模型與相關元件
    MODEL_STATE.clear()
    MODEL_STATE.update(
        {
            "model": model_data["model"],                         # 訓練好的迴歸模型
            "oe": model_data["oe"],                               # OrdinalEncoder（序數編碼器）
            "ohe": model_data["ohe"],                             # OneHotEncoder（獨熱編碼器）
            "scaler": model_data["scaler"],                       # 標準化器
            "r2": model_data.get("r2"),                           # R-squared 分數
            "feature_names": model_data["feature_names"],         # 特徵名稱清單
            "feature_coefs": model_data.get("feature_coefs",{}),  # 特徵與係數的對應
            "model_type": model_data.get("model_type"),           # 模型類型
            "alpha": model_data.get("alpha")                      # 正則化強度
        }
    )
    # 印出更新後的模型資訊，方便確認目前使用的模型
    print(f"✅ MODEL_STATE 已成功更新！當前模型：{MODEL_STATE['model_type']}，R² Score：{MODEL_STATE['r2']:.4f}")

load_model_state()

✅ MODEL_STATE 已成功更新！當前模型：Ridge，R² Score：0.8254


流程串成 train_api 函數

In [5]:
# 流程串成 train_api 函數
# 將「重新訓練 + 重新載入模型」兩段流程包裝成一個可重用的函式，
# 供 FastAPI 端點呼叫使用。
from fastapi import HTTPException

def train_api(config:TrainConfig) -> dict:
    """
    訓練端點：傳入測試集比例、隨機種子、模型類型與 alpha，線上重新訓練模型，並即時更新服務所使用的模型。
    """
    try:
        # 1. 執行重新訓練並儲存模型（會覆寫原有的 salary_model.joblib）
        res = train_and_save_model(
            test_size=config.test_size,        # 測試集比例
            random_state= config.random_state, # 隨機種子
            model_type= config.model_type,     # 模型類型
            alpha=config.alpha                 # 正則化強度
        )
         # 2. 線上重新載入最新模型狀態至全域變數，確保服務立即使用新模型
        load_model_state()
    except Exception as e:
        # 若訓練過程發生任何例外，回傳 HTTP 500 與錯誤訊息
        raise HTTPException(status_code=500, detail=f"線上訓練失敗: {str(e)}")

    return res

FastAPI TestClient 整合測試

In [6]:
# FastAPI TestClient 整合測試
# 建立一個小型 FastAPI 應用程式，並透過 TestClient 模擬實際發送 HTTP 請求，
# 驗證 /train 端點能正確完成重訓流程。
from fastapi import FastAPI
from fastapi.testclient import TestClient

# 建立迷你 FastAPI 應用程式
mini_api = FastAPI()
# 註冊 POST /train 端點，回應格式以 TrainResult 進行驗證
@mini_api.post("/train", response_model=TrainResult)
def train_endpoint(config:TrainConfig):
    # 呼叫前面包裝好的 train_api 進行實際訓練
    res = train_api(config=config)
    return res

# 建立測試用的客戶端（不需實際啟動伺服器）
client = TestClient(mini_api)
# 模擬前端傳入訓練參數，請求 /train 端點
response = client.post("/train", json={
    "test_size": 0.2,          # 測試集比例
    "random_state": 76,        # 隨機種子
    "model_type": "Lasso",     # 使用 Lasso 迴歸
    "alpha": 5.0               # 正則化強度
})
print("【重訓 Lasso 結果】")
print("HTTP 狀態碼:", response.status_code)   # 印出 HTTP 回應狀態碼
pprint(response.json())                         # 以易讀格式印出回應內容

開始訓練 Lasso 迴歸(α=5.0) (測試集比例:0.2, 隨機種子:76)....
正在將模型、預處理器與元數據序列化並儲存至 c:\Users\User\Documents\GitHub\2027-07-03clone\backend\2026_08_07\salary_model.joblib...
模型儲存成功！
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
【重訓 Lasso 結果】
HTTP 狀態碼: 200
{'alpha': 5.0,
 'coef': [0.21476373685969363, 11.506443086096304, -0.0, -0.0, 0.0],
 'feature_coefs': {'City_城市A': -0.0,
                   'City_城市B': -0.0,
                   'City_城市C': 0.0,
                   'EducationLevel': 11.506443086096304,
                   'YearsExperience': 0.21476373685969363},
 'intercept': 51.228571428571435,
 'message': 'Lasso 迴歸(α=5.0) 模型訓練完成並儲存成功！',
 'model_type': 'Lasso',
 'r2': 0.8441175875255694,
 'status': 'success',
 'train_time': 0.006991386413574219}


c:\Users\User\Documents\GitHub\2027-07-03clone\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
